# GalaxyLens-Robust — Colab runner

This notebook automates the first implementation run in Google Colab. It downloads the public dataset from Zenodo, extracts the implementation ZIP you upload, inspects the dataset, creates the fixed split, runs unit tests and a one-epoch smoke test. Then you can choose to launch the baseline and robustness evaluations.

**Important:** runtime and GPU availability depend on your Colab session. Keep the session alive during training. This notebook does not claim results until the cells actually complete.


## 1. Upload the implementation ZIP

Download `GalaxyLens_Robust_Implementation_Stage1.zip` from this chat to your computer, then run the next cell and select that ZIP.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_candidates, 'Upload the GalaxyLens implementation ZIP.'
PROJECT_ZIP = zip_candidates[0]
print('Uploaded:', PROJECT_ZIP)


In [ ]:
import os, zipfile, pathlib, shutil
WORK = pathlib.Path('/content/GalaxyLens')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(PROJECT_ZIP) as zf: zf.extractall(WORK)
PROJECT = WORK / 'galaxylens'
assert (PROJECT / 'scripts' / 'inspect_dataset.py').exists(), 'ZIP structure unexpected; expected galaxylens/scripts/inspect_dataset.py'
os.chdir(PROJECT)
print('Project root:', PROJECT)


## 2. Install dependencies

Colab often includes PyTorch already. This cell installs the project's Python package and supporting dependencies without reinstalling torch/torchvision, reducing the chance of breaking Colab's preinstalled CUDA stack.


In [ ]:
%pip install -q numpy h5py Pillow scikit-learn matplotlib PyYAML tqdm pytest
%pip install -q -e .
import torch, torchvision, h5py
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


## 3. Download Galaxy10 DECaLS

The official HDF5 dataset is about 2.7 GB. The cell verifies the published SHA-256 before proceeding. If Zenodo is temporarily unavailable, retry the cell later rather than using an unverified file.


In [ ]:
from pathlib import Path
import hashlib, urllib.request, time
DATA = Path('data/raw/Galaxy10_DECals.h5')
DATA.parent.mkdir(parents=True, exist_ok=True)
URL = 'https://zenodo.org/records/10845026/files/Galaxy10_DECals.h5?download=1'
EXPECTED_SHA256 = '19AEFC477C41BB7F77FF07599A6B82A038DC042F889A111B0D4D98BB755C1571'
if not DATA.exists():
    print('Downloading ~2.7 GB. Keep the Colab runtime connected until it finishes.')
    urllib.request.urlretrieve(URL, DATA)
def sha256_file(path, chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size), b''): h.update(chunk)
    return h.hexdigest().upper()
actual = sha256_file(DATA)
print('SHA-256:', actual)
assert actual == EXPECTED_SHA256, 'Dataset checksum mismatch; do not continue with this file.'
print('Dataset checksum verified.')


## 4. Inspect dataset and freeze splits

Review the printed inspection output before proceeding. The expected release has 17,736 RGB images, 256×256 pixels, ten labels numbered 0–9, and HDF5 keys `images` and `ans`.


In [ ]:
!python scripts/inspect_dataset.py
!python scripts/create_splits.py --seed 42 --out data/splits/split_seed42.json


## 5. Run automated tests


In [ ]:
!pytest -q


## 6. Run a one-epoch smoke test

This verifies end-to-end loading/training/checkpoint saving. It is not a scientific result. If pretrained weights cannot download, add `--no-pretrained`.


In [ ]:
!python scripts/train.py --epochs 1 --batch-size 16 --workers 2 --out artifacts/checkpoints/smoke_best.pt --reports reports/smoke


## 7. Launch the baseline experiment (optional, after smoke test passes)

This uses validation macro-F1 for checkpoint selection. The test split is not used during training. Depending on runtime/GPU, this can take a while.


In [ ]:
RUN_FULL_BASELINE = False  # Change to True only after the smoke test succeeds.
if RUN_FULL_BASELINE:
    !python scripts/train.py --epochs 12 --batch-size 32 --workers 2 --out artifacts/checkpoints/baseline_best.pt --reports reports/baseline
    !python scripts/evaluate.py --checkpoint artifacts/checkpoints/baseline_best.pt --out reports/baseline/test_metrics.json
    !python scripts/robustness_eval.py --checkpoint artifacts/checkpoints/baseline_best.pt --out reports/baseline/robustness_metrics.json
    !python scripts/selective_eval.py --checkpoint artifacts/checkpoints/baseline_best.pt --out reports/baseline/selective_metrics.json
else:
    print('Baseline training skipped. Set RUN_FULL_BASELINE=True after reviewing the smoke test.')


## 8. Export results and logs

Download the output bundle and share it back in this chat. It contains the inspection report, split manifest, tests' output if saved separately, model checkpoints, training history and metrics. Checkpoints can be large.


In [ ]:
from datetime import datetime
OUT = pathlib.Path('/content/GalaxyLens_run_outputs.zip')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['reports', 'data/splits', 'artifacts/checkpoints']:
        p=PROJECT/folder
        if p.exists():
            for f in p.rglob('*'):
                if f.is_file() and f.name != '.gitkeep': zf.write(f, f.relative_to(PROJECT))
print('Created:', OUT, 'size MB:', round(OUT.stat().st_size/1e6, 2))
from google.colab import files
files.download(str(OUT))
